In [12]:
import joblib
import pandas as pd

In [2]:
def Default_Predictor(client_s_data):
    
    client_s_data["earliest_cr_line"] = pd.to_datetime(client_s_data["earliest_cr_line"], format="%b-%Y", errors="coerce")
    client_s_data['credit_history_years'] = (pd.Timestamp("today") - client_s_data["earliest_cr_line"]).dt.days/365
    client_s_data['credit_history_years'] = client_s_data['credit_history_years'].round(1)

    client_s_data["term"] = (client_s_data["term"].astype(str).str.extract(r"(\d+)").astype(int))

    model = joblib.load('default_pred_rf.pkl')

    Prediction = model.predict(client_s_data)

    if Prediction == [0]:
        print(Prediction, "- Likely to not default")

    elif Prediction == [1]:
        print(Prediction, "- Likely to default")


In [13]:
client_data = pd.DataFrame({
    "loan_status": ["Current"],
    "loan_amnt": [20000000],
    "term": ["48 months"],
    "int_rate": [50.25],
    "installment": [360000.00],
    "annual_inc": [500000],
    "dti": [30],
    "grade": ["E"],
    "open_acc": [12.0],
    "revol_util": [8.3],
    "earliest_cr_line": ["Mar-2012"]
})

Default_Predictor(client_data)

[0] - Likely to not default


-------

We have applied our model to a 'live' client and it works. Let's have some fun! Let's create a useable tool for people to input their metrics and get a result of whether our financial institution will see them as likely to default or not...

-------

In [4]:
import ipywidgets as wid
from IPython.display import display

In [5]:
features = [
    "loan_status",
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    "annual_inc",
    "dti",
    "grade",
    "open_acc",
    "revol_util",
    "earliest_cr_line"
]

In [6]:
# Inputs

loan_amnt = wid.FloatText(description = "Loan Amount")
int_rate = wid.FloatText(description = "Interest Rate")
installment = wid.FloatText(description = "Installment")
annual_inc = wid.FloatText(description = "Annual Income")
dti = wid.FloatText(description = "Debt To Income Ratio")
open_acc = wid.FloatText(description = "Open Lines of Credit")
revol_util = wid.FloatText(description = "Revolving Line Utilization Rate")

term = wid.Dropdown(
    options = ["","12 months", "24 months", "36 months", "48 months", "60 months", "72 months", "84 months", "96 months", "108 months", "120 months"],
    description = "Loan Term"
)

loan_status = wid.Dropdown(
    options = ["","Fully Paid", "Current", "Charged Off", "Late (31-120 days)", "In Grace Period", "Late (16-30 days)", "Does not meet the credit policy. Status:Fully Paid", "Does not meet the credit policy. Status:Charged Off", "Default"],
    description = "Loan Status"
)

grade = wid.Dropdown(
    options = ["","A", "B", "C", "D", "E", "F", "G"],
    description = "Credit Grade"
)

earliest_cr_line = wid.DatePicker(description = "Earliest Credit Line date")
#earliest_cr_line = earliest_cr_line.value.strftime("%b-%Y")


predict_button = wid.Button(description="Predict")
output = wid.Output()

In [7]:
def Preprocess_input():
    data = {
        "loan_amnt": loan_amnt.value,
        "int_rate": int_rate.value,
        "installment": installment.value,
        "annual_inc": annual_inc.value,
        "dti": dti.value,
        "open_acc": open_acc.value,
        "revol_util": revol_util.value,
        "term": term.value,
        "loan_status": loan_status.value,
        "grade": grade.value,
        "earliest_cr_line": earliest_cr_line.value.strftime("%b-%Y")
    }

    dataframe = pd.DataFrame([data])

    return dataframe

In [17]:
def Default_Predictor_2(b):
    with output:
        output.clear_output()

        dataframe = Preprocess_input()
    
        dataframe["earliest_cr_line"] = pd.to_datetime(dataframe["earliest_cr_line"], format="%b-%Y", errors="coerce")
        dataframe['credit_history_years'] = (pd.Timestamp("today") - dataframe["earliest_cr_line"]).dt.days/365
        dataframe['credit_history_years'] = dataframe['credit_history_years'].round(1)

        dataframe["term"] = (dataframe["term"].astype(str).str.extract(r"(\d+)").astype(int))

        model = joblib.load('default_pred_rf.pkl')

        Prediction = model.predict(dataframe)

        if Prediction == [0]:
            print(Prediction, "- Likely to not default")

        elif Prediction == [1]:
            print(Prediction, "- Likely to default")

In [18]:
predict_button.on_click(Default_Predictor_2)

In [19]:
display(
    loan_amnt,
    int_rate,
    installment,
    annual_inc,
    dti,
    open_acc,
    revol_util,
    term,
    loan_status,
    grade,
    earliest_cr_line,
    predict_button,
    output
)

FloatText(value=2000000.0, description='Loan Amount')

FloatText(value=22.5, description='Interest Rate')

FloatText(value=60000.0, description='Installment')

FloatText(value=300000.0, description='Annual Income')

FloatText(value=18.0, description='Debt To Income Ratio')

...

Dropdown(description='Loan Status', index=2, options=('', 'Fully Paid', 'Current', 'Charged Off', 'Late (31-12…

Dropdown(description='Credit Grade', index=5, options=('', 'A', 'B', 'C', 'D', 'E', 'F', 'G'), value='E')

DatePicker(value=datetime.date(2015, 3, 12), description='Earliest Credit Line date', step=1)

Button(description='Predict', style=ButtonStyle())

Output()

-------

Ta-daa! Now anyone can input their metrics for a desired loan they wish to get. This tool will enable them to:
- See how varioius loan providers view them from behind the desk
- Play with the metrics and see what numbers can take them from being a "likely to default" client to one that is not
- Be able to set targets for themselves to reach in order to be more confident when applying for loans

-------

Ethical use for a tool such as this will be of high importance in determining whether it can be launched for the general public. This model is not perfect. Work still needs to be done to improve its performance to a level where the output will be usable. I am open to collaborate with all bright minds to take this model to the next level and develop methodologies to ensure that this tool will be used ethically in the future. 